In [ ]:
# Importing .txt file.

from pathlib import Path

def load_text_file(file_path):
    path = Path(file_path)

    if not path.exists():
        raise FileNotFoundError(f"File not found: {file_path}")

    return path.read_text(encoding="utf-8")

text = load_text_file("../data/sample.txt")

print(type(text))
print(len(text))
print(text[:500])

In [ ]:
# Chunk creation function: space between paragraphs used for separation. 
# Source allows us to know source file name as metadata.

def chunk_text_by_paragraphs(
    text,
    source,
    max_chunk_size=1200
):
    paragraphs = [
        paragraph.strip()
        for paragraph in text.split("\n\n")
        if paragraph.strip()
    ]

    chunks = []
    current_chunk = ""

    for paragraph in paragraphs:

        if len(current_chunk) + len(paragraph) <= max_chunk_size:
            current_chunk += paragraph + "\n\n"

        else:
            if current_chunk:
                chunks.append({
                    "chunk_id": len(chunks),
                    "source": source,
                    "text": current_chunk.strip()
                })

            current_chunk = paragraph + "\n\n"

    if current_chunk:
        chunks.append({
            "chunk_id": len(chunks),
            "source": source,
            "text": current_chunk.strip()
        })

    return chunks

chunks = chunk_text_by_paragraphs(
    text=text,
    source="sample.txt"
)

print(f"Number of chunks: {len(chunks)}")
print(chunks[0])

In [ ]:
import numpy as np
from ollama import embed

# Chunk embedding with ollama model.

def create_embedding(
    text,
    model="nomic-embed-text"
):
    response = embed(
        model=model,
        input=text
    )

    return np.array(
        response["embeddings"][0],
        dtype=np.float32
    )

def embed_chunks(chunks, model="nomic-embed-text"):
    embedded_chunks = []

    for chunk in chunks:
        embedding = create_embedding(
            chunk["text"],
            model=model
        )

        embedded_chunks.append({
            **chunk,
            "embedding": embedding
        })

    return embedded_chunks

embedded_chunks = embed_chunks(chunks)

print(f"Embedded chunks: {len(embedded_chunks)}")
print(embedded_chunks[0].keys())
print(embedded_chunks[0]["embedding"].shape)

In [ ]:
# Cosine similarity definition: calculates vectors orientation in a multi-dimensional space, regardless of their magnitude. 
# Two concepts are semantically similar when their vectors are oriented in the same direction and the cosine similarity is close to 1.

def cosine_similarity(vector_a, vector_b):
    return np.dot(vector_a, vector_b) / (
        np.linalg.norm(vector_a) * np.linalg.norm(vector_b)
    )

In [ ]:
# Retrieval-Augmented Generation is a technique that allows LLMs to search for infos in external sources before generating a response.
# 1. R (Retrieval): Following the query, infos are searched in the external source using semantic similarity to find a match.

def retrieve_chunks(
    question,
    embedded_chunks,
    top_k=3,
    model="nomic-embed-text"
):
    question_embedding = create_embedding(
        question,
        model=model
    )

    results = []

    for chunk in embedded_chunks:
        score = cosine_similarity(
            question_embedding,
            chunk["embedding"]
        )

        results.append({
            "chunk_id": chunk["chunk_id"],
            "source": chunk["source"],
            "text": chunk["text"],
            "score": float(score)
        })

    results.sort(
        key=lambda x: x["score"],
        reverse=True
    )

    return results[:top_k]

# Example question and R usage.
question = "How many days per week may eligible employees work remotely?"

retrieved_chunks = retrieve_chunks(
    question,
    embedded_chunks,
    top_k=3
)

def format_sources(retrieved_chunks):
    sources = []

    for chunk in retrieved_chunks:
        sources.append({
            "source": chunk["source"],
            "chunk_id": chunk["chunk_id"],
            "score": round(chunk["score"], 4)
        })

    return sources

sources = format_sources(retrieved_chunks)
print(sources)

In [ ]:
# 2. A (Augmentation): Matched text is added to the question to provide LLM with the right context

def build_context(retrieved_chunks):
    context_parts = []

    for chunk in retrieved_chunks:
        context_part = (
            f"[Source: {chunk['source']} | Chunk: {chunk['chunk_id']}]\n"
            f"{chunk['text']}"
        )

        context_parts.append(context_part)

    return "\n\n---\n\n".join(context_parts)

context = build_context(retrieved_chunks)
print(context)

# Prompt augmentation

INSUFFICIENT_CONTEXT_MESSAGE = (
    "The provided documents do not contain enough "
    "information to answer this question."
)

SYSTEM_PROMPT = f"""
You are a document analysis assistant.
Your task is to answer questions based only on the provided document context.
Rules:
- Use only the supplied context.
- Do not invent facts or unsupported explanations.
- Do not use external knowledge.
- If the context is insufficient, respond exactly with:
  "{INSUFFICIENT_CONTEXT_MESSAGE}"
- Keep answers clear, concise and factual.
""".strip()

def build_user_prompt(question, retrieved_chunks):
    context = build_context(retrieved_chunks)
    return f"""
    DOCUMENT CONTEXT {context}
    QUESTION {question}
    """.strip()

user_prompt = build_user_prompt(
    question,
    retrieved_chunks
)

print(user_prompt)

# Test no context

question_out_of_scope = "What is the company's annual revenue?"

retrieved_out_of_scope = retrieve_chunks(
    question_out_of_scope,
    embedded_chunks,
    top_k=3
)

prompt_out_of_scope = build_user_prompt(
    question_out_of_scope,
    retrieved_out_of_scope
)

print(prompt_out_of_scope)

In [ ]:
from ollama import chat

# 3. G (Generation): LLM reads the question and all the retrieved infos, then write an answer based on factual and verifiable data.

def generate_answer(
    question,
    retrieved_chunks,
    model="qwen3.5:4b"
):
    user_prompt = build_user_prompt(
        question,
        retrieved_chunks
    )

    response = chat(
        model=model,
        messages=[
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": user_prompt
            }
        ],
        think=False,
        options={
            "temperature": 0
        }
    )

    return response["message"]["content"].strip()

question = "How many days per week may eligible employees work remotely?"

retrieved_chunks = retrieve_chunks(
    question,
    embedded_chunks,
    top_k=3
)

answer = generate_answer(
    question,
    retrieved_chunks
)

print(answer)

# Ollama doesn't perform any search; retrieved_chunks are passed to build_user_prompt, so the LLM receives the question 
# and the document context and then generates a response.

# Test no context.

question_out_of_scope = "What is the company's annual revenue?"

retrieved_out_of_scope = retrieve_chunks(
    question_out_of_scope,
    embedded_chunks,
    top_k=3
)

answer_out_of_scope = generate_answer(
    question_out_of_scope,
    retrieved_out_of_scope
)

print(answer_out_of_scope)

In [ ]:
from pydantic import BaseModel
from typing import List

# Structuring output with Pydantic: class definition for the schemas.

class SourceReference(BaseModel):
    source: str
    chunk_id: int
    score: float


class DocumentAnswer(BaseModel):
    question: str
    answer: str
    answerable: bool
    retrieved_sources: List[SourceReference]

# Dict conversion in Pydantic objects.

def format_sources(retrieved_chunks):
    return [
        SourceReference(
            source=chunk["source"],
            chunk_id=chunk["chunk_id"],
            score=round(chunk["score"], 4)
        )
        for chunk in retrieved_chunks
    ]

In [ ]:
# End to end version: RAG (executes every single stage: R+A+G); generate_answer assumes that R and A stages have already been completed.
# The threshold avoids low-relevance LLM calls; it must be calibrated for each corpus.

def ask_document(
    question,
    embedded_chunks,
    top_k=3,
    min_score=0.45,
    embedding_model="nomic-embed-text",
    llm_model="qwen3.5:4b"
):
    retrieved_chunks = retrieve_chunks(
        question,
        embedded_chunks,
        top_k=top_k,
        model=embedding_model
    )

    sources = format_sources(retrieved_chunks)

    # No retrieved chunks
    if not retrieved_chunks:
        return DocumentAnswer(
            question=question,
            answer=INSUFFICIENT_CONTEXT_MESSAGE,
            answerable=False,
            retrieved_sources=[]
        )

    # Retrieval relevance check
    best_score = retrieved_chunks[0]["score"]

    if best_score < min_score:
        return DocumentAnswer(
            question=question,
            answer=INSUFFICIENT_CONTEXT_MESSAGE,
            answerable=False,
            retrieved_sources=sources
        )

    # Generation
    answer = generate_answer(
        question,
        retrieved_chunks,
        model=llm_model
    )

    # The retrieval passed the threshold, but the LLM
    # determined that the context is still insufficient.
    answerable = (
        answer.strip().lower()
        != INSUFFICIENT_CONTEXT_MESSAGE.lower()
    )

    return DocumentAnswer(
        question=question,
        answer=answer,
        answerable=answerable,
        retrieved_sources=sources
    )

# in scope

result_scope = ask_document(
    "How many days per week may eligible employees work remotely?",
    embedded_chunks
)

print(result_scope.model_dump())

# out of scope

result_no_scope = ask_document(
    "What is the company's annual revenue?",
    embedded_chunks
)

print(result_no_scope.model_dump())

In [ ]:
def print_result(result):
    print("QUESTION")
    print(result.question)

    print("\nANSWERABLE")
    print(result.answerable)

    print("\nANSWER")
    print(result.answer)

    print("\nRETRIEVED SOURCES")

    if not result.retrieved_sources:
        print("No relevant sources found.")
        return

    for source in result.retrieved_sources:
        print(
            f"- {source.source} "
            f"| Chunk {source.chunk_id} "
            f"| Score {source.score}"
        )

print_result(result_scope)
print_result(result_no_scope)

In [ ]:
# Output export

def save_result(result, output_path):
    path = Path(output_path)

    path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    path.write_text(
        result.model_dump_json(indent=2),
        encoding="utf-8"
    )

save_result(
    result_scope,
    "../outputs/sample_answer.json"
)